<a href="https://colab.research.google.com/github/jagadeesh-usd/composer-classification/blob/jag-dev/notebooks/01_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Purpose: Load MIDI files, process them into sequences, and save for training.

In [ ]:
# 1. Import Required Libraries
import os
import time
import glob
import pickle
from multiprocessing import Pool, cpu_count
from collections import Counter

from music21 import converter, instrument, note, chord

In [ ]:
# 2. Configuration
DATA_PATH = "/content/drive/My Drive/composer_project/data/midi_files/train"
COMPOSERS = ["Bach", "byrd", "Chopin", "Handel", "Mozart", "schumann"]
SEQUENCE_LENGTH = 100 # The length of a single training sequence
VOCAB_SIZE = 2000 # The number of unique notes/chords to keep

In [ ]:
# 3. Data Processing Functions

def get_midi_files(data_path, composers):
    """Finds all MIDI files for the specified composers."""
    midi_files_dict = {}
    print(f"Searching in base path: {data_path}\n")
    for composer in composers:
        composer_path = os.path.join(data_path, composer.lower())
        if os.path.isdir(composer_path):
            search_pattern = os.path.join(composer_path, '*.mid*')
            files = glob.glob(search_pattern)
            midi_files_dict[composer] = files
            print(f"Found {len(files)} files for {composer}")
        else:
            print(f"Warning: Directory not found for {composer}: {composer_path}")
            midi_files_dict[composer] = []
    return midi_files_dict

def extract_notes(file_path):
    """Extracts notes and chords from a single MIDI file."""
    notes = []
    try:
        midi = converter.parse(file_path)
        parts = instrument.partitionByInstrument(midi)
        if parts:
            notes_to_parse = parts.parts[0].recurse()
        else:
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))
    except Exception as e:
        print(f"  Error processing {file_path}: {e}")
    return notes

In [ ]:
# 4. Main Processing Logic

# Get the dictionary of file paths
midi_files = get_midi_files(DATA_PATH, COMPOSERS)

# Flatten the list of all files to be processed
all_files_to_process = [file for files in midi_files.values() for file in files]

print(f"\nProcessing {len(all_files_to_process)} MIDI files using parallel processing...")
start_time = time.time()

# Use a pool of workers (one for each CPU core) to process files in parallel
with Pool(cpu_count()) as p:
    all_notes_list = p.map(extract_notes, all_files_to_process)

end_time = time.time()
print(f"Parallel processing finished in {end_time - start_time:.2f} seconds.")

sequences = []
labels = []
all_notes = []

print("\nCreating sequences from processed notes...")
for composer, files in midi_files.items():
    for file in files:
        try:
            # Get the index of the current file in the master list
            idx = all_files_to_process.index(file)
            notes = all_notes_list[idx]

            if len(notes) > SEQUENCE_LENGTH:
                all_notes.extend(notes)
                for i in range(0, len(notes) - SEQUENCE_LENGTH, 1):
                    sequences.append(notes[i: i + SEQUENCE_LENGTH])
                    labels.append(composer)
        except ValueError:
            continue


print(f"\nTotal sequences created: {len(sequences)}")
print(f"Total notes extracted: {len(all_notes)}")


Searching in base path: /content/drive/My Drive/composer_project/data/midi_files/train

Found 42 files for Bach
Found 42 files for byrd
Found 41 files for Chopin
Found 41 files for Handel
Found 41 files for Mozart
Found 38 files for schumann

Processing 245 MIDI files using parallel processing...


/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, channel=None, data=b'Copyright \xa9 1998'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, channel=None, data=b'Copyright \xa9 1998'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, channel=None, data=b'Copyright \xa9 1997 by'>; getting generic Instrument
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/music21/midi/translate.py:874: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=12, channel=None, da

Parallel processing finished in 231.17 seconds.

Creating sequences from processed notes...

Total sequences created: 259206
Total notes extracted: 276906


In [7]:
# 5. Create Vocabulary
print("\nCreating vocabulary...")
note_counts = Counter(all_notes)
most_common_notes = [n for n, c in note_counts.most_common(VOCAB_SIZE - 1)]
most_common_notes.append('UNK') # Add a token for unknown notes

note_to_int = {note: i for i, note in enumerate(most_common_notes)}
int_to_note = {i: note for note, i in note_to_int.items()}

print(f"Vocabulary size: {len(note_to_int)}")


Creating vocabulary...
Vocabulary size: 952


In [9]:
# 6. Save Processed Data
print("\nSaving processed data...")
SAVE_DIR = "/content/drive/My Drive/composer_project/processed_data"
os.makedirs(SAVE_DIR, exist_ok=True)

with open(os.path.join(SAVE_DIR, 'sequences.pkl'), 'wb') as f:
    pickle.dump(sequences, f)

with open(os.path.join(SAVE_DIR, 'labels.pkl'), 'wb') as f:
    pickle.dump(labels, f)

preprocessing_data = {
    'note_to_int': note_to_int,
    'int_to_note': int_to_note,
    'composers': COMPOSERS,
    'sequence_length': SEQUENCE_LENGTH,
    'vocab_size': VOCAB_SIZE
}
with open(os.path.join(SAVE_DIR, 'preprocessing_data.pkl'), 'wb') as f:
    pickle.dump(preprocessing_data, f)

print(f"Data saved to {SAVE_DIR}. Preprocessing complete.")


Saving processed data...
Data saved to /content/drive/My Drive/composer_project/processed_data. Preprocessing complete.
